## 06_test_flw_fedavg.ipynb
## Evaluación de la orquestación federada mediante Flower

Este notebook explora la sustitución de la orquestación manual desarrollada en experimentos anteriores por los mecanismos nativos de coordinación proporcionados por Flower. El objetivo es analizar la viabilidad de integrar la infraestructura experimental de aprendizaje federado y monitorización energética dentro de un framework específico para Federated Learning.

Para ello se incorpora la arquitectura cliente-servidor de Flower, incluyendo estrategias basadas en FedAvg, la creación automática de clientes y la simulación de múltiples rondas de entrenamiento federado. La integración mantiene la instrumentación energética mediante CodeCarbon con el fin de verificar que las métricas de sostenibilidad continúan siendo accesibles dentro de un entorno federado gestionado por un framework externo.

Esta fase permite evaluar las ventajas e inconvenientes del uso de Flower frente a la implementación manual previamente desarrollada, prestando especial atención a aspectos como la complejidad de integración, los requisitos computacionales, la escalabilidad y la compatibilidad con la recogida de métricas necesarias para el análisis de sostenibilidad.

Lo que aporta 06_test_flw_fedavg.ipynb es la parte de orquestración de Flower

✅ Validación de la integración con la arquitectura moderna de Flower.

✅ Validación de:

Python
1
ClientApp
2
ServerApp
3
ServerConfig
4
FedAvg


✅ Confirmación de que Flower es capaz de:

Plain Text
1
Crear clientes
2
Distribuir parámetros
3
Ejecutar FedAvg
4
Gestionar rondas


✅ Confirmación de interoperabilidad con CodeCarbon.

✅ Evidencia experimental de que el framework fue evaluado y su uso fue técnicamente viable.

0. Constantes e Imports

In [1]:
DATA_DIR = "../data"

DATASET = "CIFAR10"

SCENARIO = "federated"
NUM_CLIENTS = 5


BATCH_SIZE = 64

EPOCHS = 5

NUM_ROUNDS = 5

LEARNING_RATE = 0.001

RANDOM_SEED = 42


In [2]:
import torch

import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torch.utils.data import random_split

from torchvision import datasets
from torchvision import transforms

import numpy as np

import flwr as fl

from codecarbon import EmissionsTracker

from flwr.clientapp import ClientApp


1. Carga del Dataset y DataLoader

In [3]:

# ==================================================
# Transformaciones
# ==================================================

transform = transforms.ToTensor()

# ==================================================
# Dataset
# ==================================================

train_dataset = datasets.CIFAR10(
    root="../data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="../data",
    train=False,
    download=True,
    transform=transform
)

# ==================================================
# Partición FL-5
# ==================================================

client_size = len(train_dataset) // NUM_CLIENTS

client_datasets = random_split(
    train_dataset,
    [client_size] * NUM_CLIENTS
)

# ==================================================
# DataLoaders de clientes
# ==================================================

client_loaders = [
    DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=True
    )
    for ds in client_datasets
]

# ==================================================
# Test loader global
# ==================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


2. Modelo

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

3. Modelo local para pruebas

In [5]:
model = SimpleCNN()

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

4. Función de entrenamiento

In [6]:
def train_model(
    model,
    train_loader,
    optimizer,
    criterion,
    epochs
):

    model.train()

    for epoch in range(epochs):

        running_loss = 0.0

        for images, labels in train_loader:

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)

        print(
            f"Epoch {epoch + 1}/{epochs} - "
            f"Loss: {epoch_loss:.4f}"
        )

5. Función de evaluación

In [7]:
def evaluate_model(
    model,
    test_loader
):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (
                predicted == labels
            ).sum().item()

    accuracy = correct / total

    return accuracy

6. Cliente Flower 

In [8]:
class FlowerClient(fl.client.NumPyClient):

    def __init__(
        self,
        model,
        train_loader,
        test_loader
    ):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader

    def get_parameters(
        self,
        config
    ):
        return [
            val.detach().numpy()
            for _, val in self.model.state_dict().items()
        ]
    
    def set_parameters(
        self,
        parameters
    ):
        
        params_dict = zip(
            self.model.state_dict().keys(),
            parameters
        )

        state_dict = {
            k: torch.tensor(v)
            for k, v in params_dict
        }

        self.model.load_state_dict(
            state_dict,
            strict=True
        )

    def fit(
        self,
        parameters,
        config
    ):
        self.set_parameters(parameters)

        local_criterion = nn.CrossEntropyLoss()

        local_optimizer = optim.Adam(
            self.model.parameters(),
            lr=LEARNING_RATE
        )

        tracker = EmissionsTracker(
            save_to_file=False
        )

        tracker.start()

        tracker.start_task("training")

        train_model(
            self.model,
            self.train_loader,
            local_optimizer,
            local_criterion,
            EPOCHS
        )

        training_data = tracker.stop_task()

        tracker.stop()

        metrics = {
            "energy_kwh": training_data.energy_consumed,
            "emissions_kg": training_data.emissions,
            "duration_s": training_data.duration
        }

        return (
            self.get_parameters(config),
            len(self.train_loader.dataset),
            metrics
        )
    
    def evaluate(
        self,
        parameters,
        config
    ):

        self.set_parameters(parameters)

        accuracy = evaluate_model(
          self.model,
          self.test_loader
        )

        return (
            0.0,
            len(self.test_loader.dataset),
            {
                "accuracy": accuracy
            }
        )

7. Estrategia Flower FedAvg

In [9]:
from flwr.server.strategy import FedAvg

strategy = FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS
)

8. Servidor Flower

In [10]:
from flwr.server import ServerConfig

server_config = ServerConfig(
    num_rounds=NUM_ROUNDS
)

9. Función de creación de clientes

In [11]:
def client_fn(context):

    partition_id = context.node_config["partition-id"]

    model = SimpleCNN()

    return FlowerClient(
        model=model,
        train_loader=client_loaders[partition_id],
        test_loader=test_loader
    ).to_client()


client_app = ClientApp(
    client_fn=client_fn
)

10.ServerApp

In [15]:
from flwr.serverapp import ServerApp
from flwr.compat.server import ServerAppComponents


def server_fn(context):

    return ServerAppComponents(
        strategy=strategy,
        config=server_config,
    )


server_app = ServerApp(
    server_fn=server_fn
)

In [16]:
fl.simulation.run_simulation(
    server_app=server_app,
    client_app=client_app,
    num_supernodes=NUM_CLIENTS
)


            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower ServerApp, config: num_rounds=5, no round_timeout
INFO :      
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
INFO :      Received initial parameters from one random client
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 5 clients (out of 5)
(ClientAppActor pid=88246) [codecarbon WARNING @ 16:43:38] Multiple instances of codecarbon are allowed to run at the same time.
(ClientAppActor pid=88246) /home/rutpal/python/lib/python3.12/site-packages/codecarbon/emissions_tracker.py:572: DeprecationWarning: The save_to_* parameters are deprecated and will be removed in a future version. Use output_methods=[OutputMethod.CSV, ...] instead.
(ClientAppActor pid=88246)   sel

(ClientAppActor pid=88248) Epoch 1/5 - Loss: 1.8785
(ClientAppActor pid=88248) Epoch 2/5 - Loss: 1.4973 [repeated 3x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)


(raylet) [2026-10-04 16:44:25,313 E 87593 87593] (raylet) node_manager.cc:3303: 2 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: 5865b58647ae7c6fb10236390a0998ae3e3de97bdcc39ca27792fe6f, IP: 192.168.68.230) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 192.168.68.230`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.
(ClientAppActor pid=88261) [codecarbon WARNING @ 16:43:44] Multiple instances of codecarbon are allowed to run at

(ClientAppActor pid=88248) Epoch 3/5 - Loss: 1.3420 [repeated 3x across cluster]
(ClientAppActor pid=88248) Epoch 4/5 - Loss: 1.2586 [repeated 3x across cluster]


(ClientAppActor pid=88248) [codecarbon INFO @ 16:44:51] Energy consumed for RAM : 0.000176 kWh. RAM Power : 10.0 W
(ClientAppActor pid=88248) [codecarbon INFO @ 16:44:51] Delta energy consumed for CPU with cpu_load : 0.000181 kWh, power : 10.319539104075002 W
(ClientAppActor pid=88248) [codecarbon INFO @ 16:44:51] Energy consumed for All CPU : 0.000182 kWh
(ClientAppActor pid=88248) [codecarbon INFO @ 16:44:51] 0.000357 kWh of electricity and 0.000000 L of water were used since the beginning.


...

(ClientAppActor pid=88248) [codecarbon INFO @ 16:50:16] Energy consumed for RAM : 0.000175 kWh. RAM Power : 10.0 W [repeated 3x across cluster]
(ClientAppActor pid=88248) [codecarbon INFO @ 16:50:16] Delta energy consumed for CPU with cpu_load : 0.000183 kWh, power : 10.466693133257145 W [repeated 3x across cluster]
(ClientAppActor pid=88248) [codecarbon INFO @ 16:50:16] Energy consumed for All CPU : 0.000183 kWh [repeated 3x across cluster]
(ClientAppActor pid=88248) [codecarbon INFO @ 16:50:16] 0.000357 kWh of electricity and 0.000000 L of water were used since the beginning. [repeated 3x across cluster]
(ClientAppActor pid=88262) [codecarbon WARNING @ 16:49:11] Multiple instances of codecarbon are allowed to run at the same time.
(ClientAppActor pid=88262) /home/rutpal/python/lib/python3.12/site-packages/codecarbon/emissions_tracker.py:572: DeprecationWarning: The save_to_* parameters are deprecated and will be removed in a future version. Use output_methods=[OutputMethod.CSV, ...] 

(ClientAppActor pid=88248) Epoch 5/5 - Loss: 0.1744 [repeated 3x across cluster]


INFO :      aggregate_fit: received 3 results and 2 failures
INFO :      configure_evaluate: strategy sampled 5 clients (out of 5)
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     An exception was raised when processing a message by RayBackend
ERROR :     1 worker(s) were killed due to the node running low on memory. Memory on the node (IP: 192.168.68.230, ID: 5865b58647ae7c6fb10236390a0998ae3e3de97bdcc39ca27792fe6f) was 7.26GB / 7.63GB (0.951489), which exceeds the memory usage threshold of 0.950000; Object store memory usage: [- objects spillable: 0; - bytes spillable: 0; - objects unsealed: 0; - bytes unsealed: 0; - objects in use: 8; - bytes in use: 747143837; - objects evictable: 0; - bytes evictable: 0; ; - objects created by worker: 8; - bytes created by worker: 747143837; - objects restored: 0; - bytes restored: 0; - objects received: 0; - bytes received: 0; - objects errored: 0; - bytes errored: 0; ; Eviction Stats:; (global lru) capacity

(ClientAppActor pid=88249) Epoch 5/5 - Loss: 0.1721 [repeated 2x across cluster]


(ClientAppActor pid=88249) [codecarbon INFO @ 16:50:17] Energy consumed for RAM : 0.000179 kWh. RAM Power : 10.0 W [repeated 2x across cluster]
(ClientAppActor pid=88249) [codecarbon INFO @ 16:50:17] Delta energy consumed for CPU with cpu_load : 0.000187 kWh, power : 10.424283190596924 W [repeated 2x across cluster]
(ClientAppActor pid=88249) [codecarbon INFO @ 16:50:17] Energy consumed for All CPU : 0.000187 kWh [repeated 2x across cluster]
(ClientAppActor pid=88249) [codecarbon INFO @ 16:50:17] 0.000366 kWh of electricity and 0.000000 L of water were used since the beginning. [repeated 2x across cluster]


In [14]:
from flwr.compat.server import ServerAppComponents
import inspect

print(
    inspect.signature(ServerAppComponents)
)

(server: 'Server | None' = None, config: 'ServerConfig | None' = None, strategy: 'Strategy | None' = None, client_manager: 'ClientManager | None' = None) -> None


7. Función FedAvg manual

In [ ]:
def fedavg(client_parameters):

    aggregated_parameters = []

    for layer_weights in zip(*client_parameters):

        aggregated_layer = np.mean(
            layer_weights,
            axis=0
        )

        aggregated_parameters.append(
            aggregated_layer
        )

    return aggregated_parameters

In [ ]:
## Obtener pesos iniciales

global_model = SimpleCNN()

global_parameters = [
    val.detach().numpy()
    for _, val in global_model.state_dict().items()
]

In [ ]:
# Crear clientes de Flower

clients = []

for i in range(NUM_CLIENTS):

    clients.append(
        FlowerClient(
            model=SimpleCNN(),
            train_loader=client_loaders[i],
            test_loader=test_loader
        )
    )

In [ ]:
# Entrenamiento federado de los 5 clientes durante 5 rondas

client_parameters = []

client_metrics = []

for round_num in range(NUM_ROUNDS):

    client_parameters = []

    client_metrics = []

    for client_id, client in enumerate(clients):

        updated_parameters, num_examples, metrics = client.fit(
            global_parameters,
            {}
        )
        metrics["round"] = round_num + 1
        metrics["client_id"] = client_id + 1
        print(
            f"Round {metrics['round']} | "
            f"Client {metrics['client_id']} | "
            f"Energy={metrics['energy_kwh']:.6f} kWh | "
            f"Emissions={metrics['emissions_kg']:.8f} kgCO2eq"
        )
        
        client_metrics.append(metrics)

        client_parameters.append(
            updated_parameters
        )
        
    global_parameters = fedavg(
        client_parameters
    )

8. Métricas federadas

In [ ]:
energy_total = sum(
    m["energy_kwh"]
    for m in client_metrics
)

emissions_total = sum(
    m["emissions_kg"]
    for m in client_metrics
)

duration_total = sum(
    m["duration_s"]
    for m in client_metrics
)

print("Energy:", energy_total)
print("Emissions:", emissions_total)
print("Duration:", duration_total)

9. Crear modelo global

In [ ]:
# Cargamos los pesos federados en el modelo global

global_model = SimpleCNN()

params_dict = zip(
    global_model.state_dict().keys(),
    global_parameters
)

state_dict = {
    k: torch.tensor(v)
    for k, v in params_dict
}

global_model.load_state_dict(
    state_dict,
    strict=True
)

In [ ]:
# Obtener accuracy del modelo global en el conjunto de prueba

global_accuracy = evaluate_model(
    global_model,
    test_loader
)

print(
    f"Global Accuracy: {global_accuracy:.4f}"
)